# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kishiagaytano/wilt/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**One row = one pseudonymized content item, aggregated over March 2026.** The warehouse fact table is finer than that: its grain is one row per `report_date`, `client_hash_id`, and `content_hash_id`. I roll those daily rows up to one row per content item, because the decision I support is "which page should an editor open first", and that decision is made per page, not per page-day. The daily grain is not wasted. It is what lets me measure position properly in section 3.

**Tables.** `fact_content_daily_performance` restricted to `month=2026-03` for behaviour, joined to `dim_content` for page metadata on `client_hash_id + content_hash_id`. `client_hash_id` travels with every row so grouped splits stay possible, but it is never a feature.

**Time window.** Features and the score come from March 2026, `2026-03-01` to `2026-03-31`. The forward outcome that validates the queue lives in the following window and gets built in ML-09. June 2026 is the last month of the panel and stays sealed, because it is the natural outcome window of any past-to-future label. Developing there would mean fitting inside my own test set.

**What I rank.** The expected-CTR shortfall defined in ML-03: how far a page's click rate falls below the impression-weighted rate of pages at comparable search positions. It is a derived score I define, not an observed label, and section 3 shows why the naive version of it cannot be used as-is.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)

# Token order: environment variable, then Colab Secret. Never typed into a cell.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN and IN_COLAB:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("No HF_TOKEN found. Set it as an env var, or as a Colab Secret named HF_TOKEN.")

import duckdb, pandas as pd, numpy as np

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
FACT = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

print("connected to the gated release; token read from the environment, never from a cell")
print(f"feature window: month={MONTH}")

connected to the gated release; token read from the environment, never from a cell
feature window: month=2026-03


## 2. Fields: feature / label / context / excluded

**Deliberate exclusion: every GA4 engagement column** (`ga4_sessions`, `ga4_engaged_sessions`, `ga4_pageviews`, `scroll_events`, and the `sessions_*` family).

The reason is measurable, not stylistic. In this month `ga4_data_available` is NULL on roughly 3.0M rows and TRUE on only about 0.4M. NULL is not FALSE: those rows mean "GA4 was never wired up for this client", not "nobody engaged". A filter written `= FALSE`, or an unguarded `fillna(0)`, silently converts millions of untracked rows into zero-engagement rows and teaches a model that a whole class of clients has disengaged readers. Query 3 counts this directly.

`gsc_data_available` behaves differently here, with no NULLs in this month, so the search side is safe to filter on. I still write `IS TRUE` rather than `= TRUE` for both, because the schema permits NULL and another month partition may well contain them.

| Bucket | Fields |
|---|---|
| **Features** (5, section 3) | `impressions_march`, `weighted_position`, `position_volatility`, `days_with_impressions`, `content_age_days` |
| **Label / target components** | `gsc_clicks`, and any CTR derived from it. Used to build the score, never fed in as an input |
| **Context** (carried, not modelled) | `client_hash_id` for grouped splits, `content_hash_id` as the key, `content_type` for reading results |
| **Excluded** | All GA4 engagement columns, for the NULL semantics above. Deleted content via `is_deleted`, since an editor cannot rewrite a page that no longer exists |

In [2]:
BUCKETS = {
    "feature": ["impressions_march", "weighted_position", "position_volatility",
                "days_with_impressions", "content_age_days"],
    "label component (never a feature)": ["gsc_clicks", "ctr_march"],
    "context (carried, not modelled)": ["client_hash_id", "content_hash_id", "content_type"],
    "excluded": ["ga4_sessions", "ga4_engaged_sessions", "ga4_pageviews", "scroll_events",
                 "sessions_organic", "sessions_ai", "is_deleted content"],
}
for bucket, fields in BUCKETS.items():
    print(f"{bucket:<35} {len(fields):>2}  {', '.join(fields)}")

assert not set(BUCKETS["feature"]) & set(BUCKETS["label component (never a feature)"]), \
    "a label component leaked into the feature list"
print("\ncheck passed: no label component appears in the feature list")

feature                              5  impressions_march, weighted_position, position_volatility, days_with_impressions, content_age_days
label component (never a feature)    2  gsc_clicks, ctr_march
context (carried, not modelled)      3  client_hash_id, content_hash_id, content_type
excluded                             7  ga4_sessions, ga4_engaged_sessions, ga4_pageviews, scroll_events, sessions_organic, sessions_ai, is_deleted content

check passed: no label component appears in the feature list


## 3. Verify it with queries (grain, counts, missing values, windows)

Three claims from sections 1 and 2, three queries. Then the five features, then the leak experiment.

**Query 1 proves the grain.** If `report_date + client_hash_id + content_hash_id` is really unique, the duplicate probe returns nothing. If it returned rows, every per-page aggregate below would double-count and I would never see it.

**Query 2 proves the slice.** Row count and date span for `month=2026-03`, so the window in my contract is the window in the data rather than the window I assumed.

**Query 3 proves availability, using `IS TRUE`.** Counted three ways for both flags, because the difference between FALSE and NULL is the difference between "measured as zero" and "never measured".

In [3]:
# --- Query 1: grain ---------------------------------------------------------
dupes = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {FACT}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"Q1 grain: duplicate (date, client, content) keys found = {len(dupes)}")
print("   an empty result means one row really is one page-day\n")

# --- Query 2: slice size and span -------------------------------------------
span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS first_day, MAX(report_date) AS last_day,
           COUNT(DISTINCT client_hash_id) AS n_clients, COUNT(DISTINCT content_hash_id) AS n_content
    FROM {FACT}
""").df()
print("Q2 slice:")
print(span.to_string(index=False), "\n")

# --- Query 3: availability, IS TRUE -----------------------------------------
avail = con.sql(f"""
    SELECT
      COUNT(*) AS n_rows,
      COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)  AS gsc_is_true,
      COUNT(*) FILTER (WHERE gsc_data_available IS FALSE) AS gsc_is_false,
      COUNT(*) FILTER (WHERE gsc_data_available IS NULL)  AS gsc_is_null,
      COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)  AS ga4_is_true,
      COUNT(*) FILTER (WHERE ga4_data_available IS FALSE) AS ga4_is_false,
      COUNT(*) FILTER (WHERE ga4_data_available IS NULL)  AS ga4_is_null
    FROM {FACT}
""").df()
print("Q3 availability:")
print(avail.T.rename(columns={0: "rows"}).to_string())
n = avail.iloc[0]
print(f"\n   rows surviving `gsc_data_available IS TRUE` : {n.gsc_is_true:,} of {n.n_rows:,}")
print(f"   GA4 NULL rows that `= FALSE` would mishandle: {n.ga4_is_null:,}")

Q1 grain: duplicate (date, client, content) keys found = 0
   an empty result means one row really is one page-day

Q2 slice:
 n_rows  first_day   last_day  n_clients  n_content
9841378 2026-03-01 2026-03-31         55     331437 

Q3 availability:
                 rows
n_rows        9841378
gsc_is_true   3611061
gsc_is_false  6230317
gsc_is_null         0
ga4_is_true    413966
ga4_is_false  6408671
ga4_is_null   3018741

   rows surviving `gsc_data_available IS TRUE` : 3,611,061 of 9,841,378
   GA4 NULL rows that `= FALSE` would mishandle: 3,018,741


In [4]:
# Five features, built from the same month. No clicks and no CTR among them.
features = con.sql(f"""
    WITH daily AS (
        SELECT client_hash_id, content_hash_id, gsc_impressions, gsc_clicks,
               gsc_sum_position, gsc_avg_position
        FROM {FACT}
        WHERE gsc_data_available IS TRUE
    ),
    agg AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions)                                    AS impressions_march,
               SUM(gsc_clicks)                                         AS clicks_march,
               SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS weighted_position,
               stddev_samp(gsc_avg_position) FILTER (WHERE gsc_impressions > 0) AS position_volatility,
               COUNT(*) FILTER (WHERE gsc_impressions > 0)             AS days_with_impressions
        FROM daily
        GROUP BY 1, 2
    )
    SELECT a.*,
           DATE '2026-03-31' - d.content_created_date AS content_age_days,
           d.content_type
    FROM agg a
    JOIN {DIM_CONTENT} d USING (client_hash_id, content_hash_id)
    WHERE a.impressions_march >= 1000
      AND d.is_deleted IS NOT TRUE
""").df()

features["position_volatility"] = features["position_volatility"].fillna(0)

AVAILABLE_WHEN = {
    "impressions_march":     "counted inside the feature window, so it is known the moment that window closes",
    "weighted_position":     "SUM(gsc_sum_position)/SUM(gsc_impressions) over the same window, an exposure-weighted mean rather than a mean of daily means",
    "position_volatility":   "day-to-day spread of position inside the window, knowable because every day in it has already happened",
    "days_with_impressions": "a count of days in the window carrying any impression, complete once the window ends",
    "content_age_days":      "window end minus content_created_date, a property of the page that predates the window entirely",
}
print(f"feature frame: {len(features):,} content items, {features['client_hash_id'].nunique()} clients\n")
for name, why in AVAILABLE_WHEN.items():
    print(f"  {name:<22} knowable at the decision moment because {why}")

features.head()

feature frame: 45,054 content items, 32 clients

  impressions_march      knowable at the decision moment because counted inside the feature window, so it is known the moment that window closes
  weighted_position      knowable at the decision moment because SUM(gsc_sum_position)/SUM(gsc_impressions) over the same window, an exposure-weighted mean rather than a mean of daily means
  position_volatility    knowable at the decision moment because day-to-day spread of position inside the window, knowable because every day in it has already happened
  days_with_impressions  knowable at the decision moment because a count of days in the window carrying any impression, complete once the window ends
  content_age_days       knowable at the decision moment because window end minus content_created_date, a property of the page that predates the window entirely


,client_hash_id,content_hash_id,impressions_march,clicks_march,weighted_position,position_volatility,days_with_impressions,content_age_days,content_type
0,client_73cda7b4e4f265ea,content_ab290c090bf70554,5423.0,30.0,3.013830,0.774749,31,410,keyword article
1,client_73cda7b4e4f265ea,content_2452cf1fb7b5364b,2302.0,5.0,2.525630,2.646996,31,410,keyword article
2,client_73cda7b4e4f265ea,content_ed3e64aa9adc9057,3813.0,11.0,4.761343,1.660753,31,410,keyword article
3,client_73cda7b4e4f265ea,content_a50299122f1272de,1983.0,1.0,11.795260,7.561680,31,410,keyword article
4,client_73cda7b4e4f265ea,content_7dc266fcc2cb8244,1463.0,9.0,3.477102,0.977146,31,410,keyword article


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

# The target: is this page below the impression-weighted CTR of its position stratum?
f = features.copy()
f["ctr_march"] = 100 * f["clicks_march"] / f["impressions_march"]
f["stratum"] = pd.cut(f["weighted_position"], [0, 3, 10, 20, 50, np.inf],
                      labels=["top_3", "page_1", "striking", "page_3_5", "deep"])

expected = (f.groupby("stratum", observed=True)[["clicks_march", "impressions_march"]].sum()
              .assign(expected_ctr=lambda d: 100 * d.clicks_march / d.impressions_march)["expected_ctr"])
f["expected_ctr"] = f["stratum"].map(expected).astype(float)
f["y_below_expected"] = (f["ctr_march"] < f["expected_ctr"]).astype(int)
print("impression-weighted expected CTR by stratum (%):")
print(expected.round(3).to_string())
print(f"\nbase rate, pages below their stratum expectation: {f['y_below_expected'].mean():.3f}\n")

FEATURES = list(AVAILABLE_WHEN)
y, groups = f["y_below_expected"], f["client_hash_id"]
train, test = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42).split(f, y, groups))

def auc(cols):
    model = RandomForestClassifier(n_estimators=120, max_depth=8, min_samples_leaf=25,
                                   random_state=42, n_jobs=-1)
    model.fit(f.iloc[train][cols], y.iloc[train])
    return roc_auc_score(y.iloc[test], model.predict_proba(f.iloc[test][cols])[:, 1])

honest = auc(FEATURES)
print(f"honest ROC-AUC, 5 features, client-grouped holdout : {honest:.3f}")

# The trap, sprung on purpose: ctr_march is what the label is computed from.
leaked = auc(FEATURES + ["ctr_march"])
print(f"leaked ROC-AUC, after adding ctr_march             : {leaked:.3f}   <- looks superb")
print(f"jump                                               : +{leaked - honest:.3f}")

del f["ctr_march"]
print(f"\nctr_march deleted. The number I keep is the honest one: {honest:.3f}")
print("The leak is not subtle maths. y_below_expected is computed from ctr_march,")
print("so handing the model ctr_march hands it the answer in a thin disguise.")

impression-weighted expected CTR by stratum (%):
stratum
top_3       0.391
page_1      0.325
striking    0.336
page_3_5    0.140
deep        0.032

base rate, pages below their stratum expectation: 0.652

honest ROC-AUC, 5 features, client-grouped holdout : 0.619
leaked ROC-AUC, after adding ctr_march             : 1.000   <- looks superb
jump                                               : +0.381

ctr_march deleted. The number I keep is the honest one: 0.619
The leak is not subtle maths. y_below_expected is computed from ctr_march,
so handing the model ctr_march hands it the answer in a thin disguise.


## 4. Data limits

**Named limitation: this slice describes a visible minority of the inventory, and a minority of clients.**

Three counts from section 3 give its shape. March 2026 holds 55 of the panel's 104 clients, so nearly half the client base contributes nothing to this window, mostly because their tracking started later. Of roughly 9.8M page-days, only about 3.6M carry search data with `gsc_data_available IS TRUE`. And of the 331,437 content items present, the median one records 173 impressions and zero clicks across the entire month, which is why the 1,000-impression floor cuts the frame down to 45,054 pages. Those pages belong to 32 clients rather than 55, because the floor removes whole clients whose pages all sit below it.

What follows from that. A queue built here speaks about pages already visible enough to measure, belonging to clients already instrumented in early 2026. It says nothing about pages with no search presence, which is a different problem needing a different method, and nothing about newly onboarded clients. Any claim I publish carries that scope.

Two further limits worth naming now rather than discovering later. The panel is unbalanced, so per-client history depth differs and a single global calendar window quietly favours long-tenured clients; per-client windows are the honest alternative. And GA4 coverage is both thin and three-valued in this month, which is why engagement plays no part in the contract even though the lane's name mentions it.

In [6]:
limits = con.sql(f"""
    SELECT
      (SELECT COUNT(*) FROM read_parquet('{REL}/dim_clients.parquet')) AS clients_in_panel,
      COUNT(DISTINCT client_hash_id)                                   AS clients_in_march,
      COUNT(DISTINCT content_hash_id)                                  AS content_in_march
    FROM {FACT}
""").df()
print(limits.to_string(index=False))

coverage = con.sql(f"""
    WITH agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS imp, SUM(gsc_clicks) AS clk
        FROM {FACT} WHERE gsc_data_available IS TRUE GROUP BY 1
    )
    SELECT COUNT(*) AS content_with_search_rows,
           median(imp) AS median_impressions,
           median(clk) AS median_clicks,
           COUNT(*) FILTER (WHERE imp >= 1000) AS above_floor
    FROM agg
""").df()
print()
print(coverage.to_string(index=False))
share = 100 * coverage.above_floor.iloc[0] / coverage.content_with_search_rows.iloc[0]
print(f"\nshare of measurable content items clearing the 1,000-impression floor: {share:.1f}%")

 clients_in_panel  clients_in_march  content_in_march
              104                55            331437

 content_with_search_rows  median_impressions  median_clicks  above_floor
                   176738               173.0            0.0        45058

share of measurable content items clearing the 1,000-impression floor: 25.5%


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.